# TechJam production BGE artifact pipeline
Build and validate the stock `BAAI/bge-base-en-v1.5` catalogue cache first. The optional section trains and evaluates a separately named model; it never changes production selection.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/nickolaschua/reptechjam2026.git')
REPO_REF = os.environ.get('REPO_REF', 'main')  # set an exact commit SHA for releases
REPO_ROOT = Path('/content/techjam26')
ARTIFACT_ROOT = Path('/content/bge_artifacts')
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_REF], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'colab' / 'requirements.txt')], check=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print('checked out', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import hashlib, inspect, json, random, sys, zipfile
import numpy as np
sys.path.insert(0, str(REPO_ROOT))

from system.shopping_agent.catalogue import Catalogue
from system.shopping_agent.embedding_backends import (
    BGEEmbeddingBackend, BGE_MODEL, BGE_QUERY_PREFIX, CacheExpectation,
    PRODUCT_TEXT_VERSION, fingerprint_file, fingerprint_texts,
    load_embedding_cache, make_embedding_space_id, production_product_texts,
    save_embedding_cache,
)
from system.shopping_agent.config import CATALOG_PATH
from sentence_transformers import SentenceTransformer

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

## Base production cache

In [ ]:
catalogue = Catalogue(CATALOG_PATH)
assert len(catalogue.ids) == 50_000, f'expected 50,000 rows, got {len(catalogue.ids):,}'
catalog_ids = tuple(catalogue.ids)
catalog_products = tuple(catalogue.products)
product_texts = production_product_texts(catalog_products)
assert len(product_texts) == len(catalog_ids) == 50_000
catalog_fingerprint = fingerprint_file(CATALOG_PATH)
product_text_fingerprint = fingerprint_texts(product_texts)
print({'rows': len(catalog_ids), 'catalog_fingerprint': catalog_fingerprint, 'product_text_fingerprint': product_text_fingerprint})

In [ ]:
model = SentenceTransformer(BGE_MODEL, device='cuda')
backend = BGEEmbeddingBackend(model=model, batch_size=256)
catalog_matrix = backend.embed_catalog(product_texts)
assert catalog_matrix.shape == (50_000, 768)
assert np.allclose(np.linalg.norm(catalog_matrix, axis=1), 1.0, rtol=1e-4, atol=1e-5)

base_relpath = Path('system/shopping_agent/embedding_cache/catalog_cache_bge-base-en-v1.5.npz')
base_cache = ARTIFACT_ROOT / base_relpath
expectation = CacheExpectation(
    backend_id=backend.backend_id, model_id=backend.model_id,
    embedding_space_id=backend.embedding_space_id, catalog_ids=catalog_ids,
    product_text_version=PRODUCT_TEXT_VERSION,
    product_text_fingerprint=product_text_fingerprint,
    catalog_fingerprint=catalog_fingerprint, vector_dimension=768, normalized=True,
)
save_embedding_cache(base_cache, catalog_matrix, expectation)
reloaded = load_embedding_cache(base_cache, expectation)
assert np.array_equal(reloaded, catalog_matrix)
query = backend.embed_query(product_texts[0])
catalog_query_cosine = float(reloaded[0] @ query)
assert np.isfinite(catalog_query_cosine) and -1.00001 <= catalog_query_cosine <= 1.00001
print({'cache': str(base_cache), 'shape': reloaded.shape, 'catalog_query_cosine': catalog_query_cosine})

In [ ]:
manifest_path = ARTIFACT_ROOT / 'bge_artifact_manifest.json'
manifest = {
    'schema_version': 1, 'repository_url': REPO_URL, 'repository_ref': REPO_REF,
    'base_model': BGE_MODEL, 'catalog_rows': 50_000, 'vector_dimension': 768,
    'catalog_fingerprint': catalog_fingerprint,
    'product_text_version': PRODUCT_TEXT_VERSION,
    'product_text_fingerprint': product_text_fingerprint,
    'catalog_query_cosine': catalog_query_cosine,
    'files': {base_cache.name: sha256_file(base_cache)},
}
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
zip_path = Path(shutil.make_archive('/content/techjam-bge-artifacts', 'zip', ARTIFACT_ROOT))
print('download:', zip_path, 'sha256:', sha256_file(zip_path))
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass

## Optional fine-tuning and fixed-holdout evaluation
This section is disabled by default. It uses the tracked messy utterance/product pairs and a deterministic product-level `random20` split because `parses.jsonl` and the historical checkpoint are not required.

In [ ]:
RUN_FINE_TUNING = False
SEED = 20260830
EPOCHS = 3
NEGATIVES_PER_ANCHOR = 2
if not RUN_FINE_TUNING:
    print('Optional fine-tuning disabled; stock BGE remains the production model.')

In [ ]:
if RUN_FINE_TUNING:
    from datasets import Dataset
    from sentence_transformers import (SentenceTransformerTrainer, SentenceTransformerTrainingArguments)
    from sentence_transformers.losses import MultipleNegativesRankingLoss

    archive_candidates = [REPO_ROOT / 'archive/winston', REPO_ROOT / 'docs/archive/winston', REPO_ROOT / 'winston']
    archive = next((path for path in archive_candidates if (path / 'lab/bench/cases.jsonl').exists()), None)
    if archive is None:
        raise FileNotFoundError('tracked messy cases were not found in either archive location')
    cases = [json.loads(line) for line in (archive / 'lab/bench/cases.jsonl').open(encoding='utf-8') if line.strip()]
    by_asin = {asin: product for asin, product in zip(catalog_ids, catalog_products)}
    row_by_asin = {asin: row for row, asin in enumerate(catalog_ids)}
    cases = [case for case in cases if case['asin'] in by_asin]
    product_ids = sorted({case['asin'] for case in cases})
    rng = random.Random(SEED)
    heldout_ids = set(rng.sample(product_ids, len(product_ids) // 5))
    train_cases = [case for case in cases if case['asin'] not in heldout_ids]
    heldout_cases = [case for case in cases if case['asin'] in heldout_ids]

    def bucket(product):
        categories = [str(value).lower() for value in product.get('categories') or []]
        return ' / '.join(categories[-2:]) if categories else 'uncategorized'

    buckets = {}
    for asin, product in by_asin.items():
        buckets.setdefault(bucket(product), []).append(asin)
    anchors, positives, negatives = [], [], []
    for case in train_cases:
        asin = case['asin']
        pool = [other for other in buckets[bucket(by_asin[asin])] if other != asin]
        if not pool:
            pool = [other for other in catalog_ids if other != asin]
        for negative_asin in rng.sample(pool, min(NEGATIVES_PER_ANCHOR, len(pool))):
            anchors.append(BGE_QUERY_PREFIX + case['utterance'])
            positives.append(product_texts[row_by_asin[asin]])
            negatives.append(product_texts[row_by_asin[negative_asin]])
    print({'cases': len(cases), 'train_cases': len(train_cases), 'heldout_cases': len(heldout_cases), 'heldout_products': len(heldout_ids), 'triplets': len(anchors)})

In [ ]:
if RUN_FINE_TUNING:
    tuned_model_dir = ARTIFACT_ROOT / 'models/bge-base-en-v1.5-techjam-random20'
    checkpoint_dir = Path('/content/bge_checkpoints')
    tuned_model = SentenceTransformer(BGE_MODEL, device='cuda')
    argument_values = dict(
        output_dir=str(checkpoint_dir), num_train_epochs=EPOCHS,
        per_device_train_batch_size=16, gradient_accumulation_steps=2,
        learning_rate=2e-5, warmup_steps=10, dataloader_num_workers=0,
        logging_steps=10, save_strategy='no', report_to='none', seed=SEED,
    )
    accepted = set(inspect.signature(SentenceTransformerTrainingArguments).parameters)
    training_args = SentenceTransformerTrainingArguments(**{key: value for key, value in argument_values.items() if key in accepted})
    training_data = Dataset.from_dict({'anchor': anchors, 'positive': positives, 'negative': negatives})
    trainer = SentenceTransformerTrainer(
        model=tuned_model, args=training_args, train_dataset=training_data,
        loss=MultipleNegativesRankingLoss(tuned_model),
    )
    trainer.train()
    tuned_model.save_pretrained(str(tuned_model_dir))

In [ ]:
if RUN_FINE_TUNING:
    tuned_matrix = BGEEmbeddingBackend(model=tuned_model, batch_size=256).embed_catalog(product_texts)

    def evaluate(model_to_test, matrix):
        reciprocal_ranks, hits = [], 0
        for case in heldout_cases:
            query_vector = model_to_test.encode(BGE_QUERY_PREFIX + case['utterance'], convert_to_numpy=True, normalize_embeddings=True)
            scores = matrix @ np.asarray(query_vector, dtype=np.float32)
            target_row = row_by_asin[case['asin']]
            rank = 1 + int(np.count_nonzero(scores > scores[target_row]))
            reciprocal_ranks.append(1.0 / rank)
            hits += rank <= 10
        return {'cases': len(heldout_cases), 'hit_at_10': hits / len(heldout_cases), 'mrr': float(np.mean(reciprocal_ranks))}

    metrics = {'seed': SEED, 'epochs': EPOCHS, 'holdout': 'product-level-random20', 'base': evaluate(model, catalog_matrix), 'tuned': evaluate(tuned_model, tuned_matrix)}
    tuned_backend_id = 'bge-base-en-v1.5-techjam-random20'
    tuned_model_id = 'BAAI/bge-base-en-v1.5+techjam-random20'
    tuned_cache = ARTIFACT_ROOT / 'system/shopping_agent/embedding_cache/catalog_cache_bge-base-en-v1.5-techjam-random20.npz'
    tuned_expectation = CacheExpectation(
        backend_id=tuned_backend_id, model_id=tuned_model_id,
        embedding_space_id=make_embedding_space_id(tuned_backend_id, tuned_model_id, 768),
        catalog_ids=catalog_ids, product_text_version=PRODUCT_TEXT_VERSION,
        product_text_fingerprint=product_text_fingerprint, catalog_fingerprint=catalog_fingerprint,
        vector_dimension=768, normalized=True,
    )
    save_embedding_cache(tuned_cache, tuned_matrix, tuned_expectation)
    load_embedding_cache(tuned_cache, tuned_expectation)
    (ARTIFACT_ROOT / 'heldout_product_ids.json').write_text(json.dumps(sorted(heldout_ids), indent=2) + '\n', encoding='utf-8')
    (ARTIFACT_ROOT / 'bge_tuning_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(metrics, indent=2))

In [ ]:
if RUN_FINE_TUNING:
    all_files = [path for path in ARTIFACT_ROOT.rglob('*') if path.is_file() and path != manifest_path]
    manifest['tuned_model_activated'] = False
    manifest['files'] = {str(path.relative_to(ARTIFACT_ROOT)).replace('\\', '/'): sha256_file(path) for path in sorted(all_files)}
    manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
    zip_path = Path(shutil.make_archive('/content/techjam-bge-artifacts-with-tuned', 'zip', ARTIFACT_ROOT))
    print('download:', zip_path, 'sha256:', sha256_file(zip_path))
    try:
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        pass
catalogue.close()